# Notebook-first application walkthrough

**Problem / objective:** Classify product/wine quality groups with an interpretable distance-based model and show why feature scaling matters for KNN.

**Decision / solution:** Route confident cases automatically and send ambiguous nearest-neighbour cases to review with the neighbour evidence visible.

This front section is intentionally analysis-first. It uses direct notebook code for inspection, EDA, visualisation and evidence review. The original notebook work is preserved below, followed by modular production code where that adds engineering evidence.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_SLUG = 'knn_product_quality'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    candidate = ROOT.parent.parent if ROOT.name == PROJECT_SLUG else ROOT
    if (candidate / 'projects').exists():
        ROOT = candidate
PROJECT = ROOT / 'projects' / PROJECT_SLUG
if not PROJECT.exists() and Path.cwd().name == PROJECT_SLUG:
    PROJECT = Path.cwd()
    ROOT = PROJECT.parent.parent
assert PROJECT.exists(), f'Project directory not found: {PROJECT}'
print('Repository root:', ROOT.resolve())
print('Project:', PROJECT.resolve())


## 1. Find the real data and retained evidence

Instead of hiding the dataset behind a helper function, start by seeing what the project actually ships: raw/small data, fixtures, outputs, results and verified evidence. External large datasets remain reproducibly downloadable from the documented source.


In [ ]:
candidate_files = []
for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
    candidate_files.extend(PROJECT.rglob(pattern))
verified_dir = ROOT / 'verified' / PROJECT_SLUG
if verified_dir.exists():
    for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
        candidate_files.extend(verified_dir.rglob(pattern))
candidate_files = sorted({p.resolve() for p in candidate_files if p.is_file()})
file_inventory = pd.DataFrame({
    'file': [str(p.relative_to(ROOT)) if ROOT in p.parents else str(p) for p in candidate_files],
    'suffix': [p.suffix.lower() for p in candidate_files],
    'size_kb': [round(p.stat().st_size / 1024, 1) for p in candidate_files],
})
display(file_inventory.head(40))
print(f'Inspectable local data/evidence files: {len(file_inventory):,}')


## 2. Direct tabular data audit

The code below deliberately avoids a project-specific wrapper. It opens the first sensible local tabular asset, shows its schema and quality profile, and makes the data issues visible before modelling. If the full raw dataset is external, run the project's documented download cell/entry point first and rerun this section.


In [ ]:
tabular_candidates = [p for p in candidate_files if p.suffix.lower() in {'.csv', '.tsv', '.parquet'}]
preferred = [p for p in tabular_candidates if not any(token in p.name.lower() for token in ('metric', 'summary', 'verification'))]
tabular_path = (preferred or tabular_candidates or [None])[0]
df = None
if tabular_path is not None:
    if tabular_path.suffix.lower() == '.parquet':
        df = pd.read_parquet(tabular_path)
    else:
        sep = '\t' if tabular_path.suffix.lower() == '.tsv' else ','
        df = pd.read_csv(tabular_path, sep=sep, nrows=200_000)
    print('Loaded:', tabular_path)
    print('Shape:', df.shape)
    display(df.head())
    audit = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': (100 * df.isna().mean()).round(2),
        'unique': df.nunique(dropna=False),
    }).sort_values(['missing_pct', 'unique'], ascending=[False, False])
    display(audit.head(30))
    print('Duplicate rows:', int(df.duplicated().sum()))
else:
    print('No local CSV/TSV/Parquet found yet. Use the project README/run path to download or build the documented dataset, then rerun this audit.')


## 3. Exploratory data analysis and visualisation

These plots are intentionally created in the notebook rather than described in prose. They expose distribution, missingness, scale, category balance and numeric relationships before any final model decision.


In [ ]:
if df is not None and len(df):
    missing_pct = (100 * df.isna().mean()).sort_values(ascending=False).head(20)
    missing_pct = missing_pct[missing_pct > 0]
    if len(missing_pct):
        plt.figure(figsize=(10, 4))
        missing_pct.plot(kind='bar')
        plt.title('Missing values by feature (%)')
        plt.ylabel('Missing %')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:8]
    for col in numeric_cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(series):
            plt.figure(figsize=(8, 4))
            plt.hist(series, bins=30, alpha=0.8)
            plt.axvline(series.median(), linestyle='--', label=f'median={series.median():.2f}')
            plt.title(f'Distribution: {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.legend()
            plt.tight_layout()
            plt.show()

    categorical_cols = [c for c in df.columns if c not in numeric_cols and df[c].nunique(dropna=False) <= 30][:4]
    for col in categorical_cols:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(15)
        plt.figure(figsize=(9, 4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Top categories: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        plt.figure(figsize=(8, 6))
        image = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
        plt.colorbar(image, label='Correlation')
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=60, ha='right')
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title('Numeric correlation matrix')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        x_col, y_col = numeric_cols[0], numeric_cols[-1]
        sample = df[[x_col, y_col]].dropna().sample(min(3000, len(df.dropna(subset=[x_col, y_col]))), random_state=42)
        if len(sample):
            plt.figure(figsize=(7, 5))
            plt.scatter(sample[x_col], sample[y_col], alpha=0.35, s=18)
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.title(f'{y_col} versus {x_col}')
            plt.tight_layout()
            plt.show()
else:
    print('Run the documented data-build/download path, then rerun this section to render raw-data EDA.')


## 4. Inspect the measured results, not just the code

A portfolio project is stronger when it retains evidence. This section reads machine-readable JSON/CSV outputs and turns scalar metrics into a quick visual comparison.


In [ ]:
json_files = [p for p in candidate_files if p.suffix.lower() == '.json']
metric_rows = []
for path in json_files[:30]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int, float)) and not isinstance(value, bool) and np.isfinite(value):
            metric_rows.append({
                'file': str(path.relative_to(ROOT)) if ROOT in path.parents else str(path),
                'metric': prefix,
                'value': float(value),
            })
metrics_df = pd.DataFrame(metric_rows)
if len(metrics_df):
    display(metrics_df.head(40))
    plot_df = metrics_df[np.isfinite(metrics_df['value'])].copy()
    plot_df = plot_df[plot_df['value'].abs() < 1_000_000].head(20)
    if len(plot_df):
        labels = (plot_df['file'].str.split('/').str[-1] + ' :: ' + plot_df['metric']).tolist()
        plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
        plt.barh(range(len(plot_df)), plot_df['value'])
        plt.yticks(range(len(plot_df)), labels)
        plt.title('Retained project metrics / evidence')
        plt.tight_layout()
        plt.show()
else:
    print('No scalar JSON evidence found. Run the project and retain metrics/results before treating it as complete.')


## 5. Reproduce the application

The notebook should be understandable without running anything, but a reviewer can reproduce the canonical application below. The switch is off by default so opening the notebook never triggers a long training job unexpectedly.


In [ ]:
RUN_PROJECT = False
entrypoint = PROJECT / 'run.py'
if RUN_PROJECT and entrypoint.exists():
    subprocess.run([sys.executable, str(entrypoint)], cwd=PROJECT, check=True)
elif entrypoint.exists():
    print(f'Reproduce with: cd {PROJECT} && {sys.executable} run.py')
else:
    print('This project uses a different documented entry point; see README.md in the project folder.')


## 6. Decision / solution

Route confident cases automatically and send ambiguous nearest-neighbour cases to review with the neighbour evidence visible.

The final recommendation should be tied to the measured validation evidence and error analysis below. A model is not the solution by itself; the solution is the decision process built around it.


# K-Nearest Neighbours — Product Quality Decision System

## Problem and objective
Build an interpretable distance-based classification application that predicts product-quality class, quantifies confidence, exposes nearest comparable observations, and sends uncertain predictions to manual review.


## Dataset and provenance
The project uses scikit-learn's built-in Wine dataset: 178 observations, 13 continuous chemical measurements and three classes. It is bundled through the sklearn dataset loader for deterministic reproduction. This is an educational quality-decision benchmark, not a claim about a live manufacturing process.


In [ ]:
from run import load_dataset, audit_dataset, descriptive_profile
x, y, target_names = load_dataset()
audit_dataset(x, y), x.head(), y.value_counts().sort_index()


## Analysis and validation
The full application uses leakage-safe scaling, stratified splitting, cross-validated KNN tuning, a scaling ablation, confusion-matrix/error analysis, permutation importance and confidence-based review logic. The canonical implementation is mirrored into this notebook by the portfolio workflow so the complete code remains visible here.


In [ ]:
from run import build_pipeline, scaling_ablation
baseline_knn = build_pipeline(n_neighbors=7)
scaling_ablation(x, y)


In [ ]:
from run import main
# Run the complete reproducible training/evaluation pipeline.
# main()


## Decision use, results and limitations
The application produces probability confidence, a manual-review flag and nearest-neighbour evidence. Results are written to `results/metrics.json` and model artefacts to `artifacts/`. Limitations include the compact benchmark size, sensitivity of distance methods to scaling and dimensionality, and the need to choose review thresholds from real operational costs. A production next step would validate on time-separated factory batches and monitor feature-distance drift.

## Reproducibility
Run `python run.py` from this project directory after installing the project requirements. Tests are in `tests/test_knn.py`.


## Interview discussion
Be ready to explain why KNN requires feature scaling, how `k` changes the bias/variance trade-off, why stratification matters, how neighbour distance can support interpretability, what the confidence-review threshold means operationally, and when a tree-based model would be a better choice.


# Engineering appendix — canonical application source

The analysis and visual evidence come first. The cells below preserve additional canonical Python from this project for reviewers who want to inspect pipelines, APIs, tests, feature code, monitoring and reusable implementation details.


## Canonical source: `run.py`


In [ ]:
from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
ROOT = Path(__file__).resolve().parent
RESULTS = ROOT / "results"
ARTIFACTS = ROOT / "artifacts"
RESULTS.mkdir(exist_ok=True)
ARTIFACTS.mkdir(exist_ok=True)


@dataclass
class DatasetAudit:
    rows: int
    columns: int
    target_classes: int
    duplicate_rows: int
    missing_cells: int
    min_class_size: int
    max_class_size: int


@dataclass
class Evaluation:
    accuracy: float
    balanced_accuracy: float
    macro_f1: float
    log_loss: float
    review_rate: float
    accepted_accuracy: float | None


def load_dataset() -> tuple[pd.DataFrame, pd.Series, list[str]]:
    bunch = load_wine(as_frame=True)
    frame = bunch.frame.copy()
    target = frame.pop("target").astype(int)
    names = [str(name) for name in bunch.target_names]
    frame.columns = [str(c).strip().lower().replace(" ", "_") for c in frame.columns]
    return frame, target, names


def audit_dataset(x: pd.DataFrame, y: pd.Series) -> DatasetAudit:
    counts = y.value_counts()
    audit = DatasetAudit(
        rows=len(x),
        columns=x.shape[1],
        target_classes=int(y.nunique()),
        duplicate_rows=int(x.duplicated().sum()),
        missing_cells=int(x.isna().sum().sum()),
        min_class_size=int(counts.min()),
        max_class_size=int(counts.max()),
    )
    if audit.rows < 100:
        raise ValueError("Dataset unexpectedly small")
    if audit.target_classes < 2:
        raise ValueError("Classification requires multiple classes")
    if audit.missing_cells:
        raise ValueError("Built-in benchmark should not contain missing values")
    return audit


def descriptive_profile(x: pd.DataFrame, y: pd.Series) -> dict[str, Any]:
    profile: dict[str, Any] = {
        "shape": [int(x.shape[0]), int(x.shape[1])],
        "class_distribution": {str(k): int(v) for k, v in y.value_counts().sort_index().items()},
        "feature_means": {k: float(v) for k, v in x.mean().items()},
        "feature_std": {k: float(v) for k, v in x.std().items()},
        "feature_min": {k: float(v) for k, v in x.min().items()},
        "feature_max": {k: float(v) for k, v in x.max().items()},
    }
    return profile


def build_pipeline(n_neighbors: int = 5, weights: str = "distance", p: int = 2) -> Pipeline:
    return Pipeline(
        steps=[
            ("scale", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, p=p)),
        ]
    )


def tune_model(x_train: pd.DataFrame, y_train: pd.Series) -> GridSearchCV:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    search = GridSearchCV(
        estimator=build_pipeline(),
        param_grid={
            "model__n_neighbors": list(range(3, 22, 2)),
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2],
        },
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        return_train_score=True,
    )
    search.fit(x_train, y_train)
    return search


def evaluate_confidence_policy(
    probabilities: np.ndarray,
    predictions: np.ndarray,
    truth: np.ndarray,
    threshold: float,
) -> tuple[float, float | None]:
    confidence = probabilities.max(axis=1)
    accepted = confidence >= threshold
    review_rate = float((~accepted).mean())
    accepted_accuracy = None
    if accepted.any():
        accepted_accuracy = float(accuracy_score(truth[accepted], predictions[accepted]))
    return review_rate, accepted_accuracy


def evaluate_model(model: Pipeline, x_test: pd.DataFrame, y_test: pd.Series) -> tuple[Evaluation, dict[str, Any]]:
    predictions = model.predict(x_test)
    probabilities = model.predict_proba(x_test)
    review_rate, accepted_accuracy = evaluate_confidence_policy(
        probabilities,
        predictions,
        y_test.to_numpy(),
        threshold=0.70,
    )
    evaluation = Evaluation(
        accuracy=float(accuracy_score(y_test, predictions)),
        balanced_accuracy=float(balanced_accuracy_score(y_test, predictions)),
        macro_f1=float(f1_score(y_test, predictions, average="macro")),
        log_loss=float(log_loss(y_test, probabilities)),
        review_rate=review_rate,
        accepted_accuracy=accepted_accuracy,
    )
    detail = {
        "confusion_matrix": confusion_matrix(y_test, predictions).tolist(),
        "classification_report": classification_report(y_test, predictions, output_dict=True),
        "confidence": probabilities.max(axis=1).tolist(),
    }
    return evaluation, detail


def scaling_ablation(x_train: pd.DataFrame, y_train: pd.Series) -> dict[str, float]:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scaled = build_pipeline(n_neighbors=7)
    unscaled = KNeighborsClassifier(n_neighbors=7, weights="distance")
    scaled_score = cross_val_score(scaled, x_train, y_train, scoring="f1_macro", cv=cv).mean()
    raw_score = cross_val_score(unscaled, x_train, y_train, scoring="f1_macro", cv=cv).mean()
    return {
        "scaled_macro_f1": float(scaled_score),
        "unscaled_macro_f1": float(raw_score),
        "scaling_gain": float(scaled_score - raw_score),
    }


def feature_importance(model: Pipeline, x_test: pd.DataFrame, y_test: pd.Series) -> list[dict[str, float | str]]:
    result = permutation_importance(
        model,
        x_test,
        y_test,
        scoring="f1_macro",
        n_repeats=25,
        random_state=RANDOM_STATE,
    )
    rows = [
        {"feature": feature, "importance": float(score)}
        for feature, score in zip(x_test.columns, result.importances_mean)
    ]
    return sorted(rows, key=lambda row: float(row["importance"]), reverse=True)


def inspect_neighbours(model: Pipeline, x: pd.DataFrame, row: pd.DataFrame, top_n: int = 5) -> list[dict[str, Any]]:
    scaler: StandardScaler = model.named_steps["scale"]
    knn: KNeighborsClassifier = model.named_steps["model"]
    x_scaled = scaler.transform(x)
    row_scaled = scaler.transform(row)
    distances, indices = knn.kneighbors(row_scaled, n_neighbors=top_n)
    output: list[dict[str, Any]] = []
    for distance, idx in zip(distances[0], indices[0]):
        output.append({"row_index": int(idx), "distance": float(distance)})
    return output


def predict_one(model: Pipeline, row: pd.DataFrame, target_names: list[str]) -> dict[str, Any]:
    probabilities = model.predict_proba(row)[0]
    prediction = int(model.predict(row)[0])
    confidence = float(probabilities.max())
    return {
        "predicted_class": prediction,
        "predicted_label": target_names[prediction],
        "confidence": confidence,
        "manual_review": bool(confidence < 0.70),
        "class_probabilities": {
            target_names[i]: float(probabilities[i]) for i in range(len(probabilities))
        },
    }


def save_json(path: Path, payload: Any) -> None:
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def main() -> None:
    x, y, target_names = load_dataset()
    audit = audit_dataset(x, y)
    profile = descriptive_profile(x, y)

    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.25,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    search = tune_model(x_train, y_train)
    best_model: Pipeline = search.best_estimator_
    evaluation, evaluation_detail = evaluate_model(best_model, x_test, y_test)
    ablation = scaling_ablation(x_train, y_train)
    importance = feature_importance(best_model, x_test, y_test)
    example = predict_one(best_model, x_test.iloc[[0]], target_names)
    neighbours = inspect_neighbours(best_model, x_train, x_test.iloc[[0]], top_n=5)

    cv_table = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
    cv_table[
        ["params", "mean_test_score", "std_test_score", "mean_train_score", "rank_test_score"]
    ].head(20).to_csv(RESULTS / "cv_results.csv", index=False)

    payload = {
        "dataset_audit": asdict(audit),
        "evaluation": asdict(evaluation),
        "best_parameters": search.best_params_,
        "best_cv_macro_f1": float(search.best_score_),
        "scaling_ablation": ablation,
        "top_permutation_features": importance[:10],
        "example_prediction": example,
        "example_neighbours": neighbours,
        "confusion_matrix": evaluation_detail["confusion_matrix"],
        "limitations": [
            "Compact benchmark dataset rather than a live production quality stream.",
            "Neighbour distances can become less informative as dimensionality grows.",
            "A production threshold should be chosen from explicit quality-review costs.",
        ],
    }
    save_json(RESULTS / "metrics.json", payload)
    save_json(RESULTS / "dataset_profile.json", profile)
    joblib.dump(best_model, ARTIFACTS / "knn_quality_pipeline.joblib")

    reloaded: Pipeline = joblib.load(ARTIFACTS / "knn_quality_pipeline.joblib")
    original_pred = best_model.predict(x_test)
    reloaded_pred = reloaded.predict(x_test)
    if not np.array_equal(original_pred, reloaded_pred):
        raise RuntimeError("Saved-model parity check failed")

    print(json.dumps(payload, indent=2))


if __name__ == "__main__":
    main()


## Canonical source: `tests/test_knn.py`


In [ ]:
from pathlib import Path
import sys

import numpy as np

PROJECT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT))

from run import audit_dataset, build_pipeline, load_dataset, predict_one


def test_dataset_contract():
    x, y, names = load_dataset()
    audit = audit_dataset(x, y)
    assert audit.rows == len(x)
    assert audit.columns == x.shape[1]
    assert audit.target_classes == 3
    assert len(names) == 3


def test_pipeline_returns_probabilities():
    x, y, names = load_dataset()
    model = build_pipeline(n_neighbors=5)
    model.fit(x, y)
    result = predict_one(model, x.iloc[[0]], names)
    probabilities = np.array(list(result["class_probabilities"].values()))
    assert np.isclose(probabilities.sum(), 1.0)
    assert 0.0 <= result["confidence"] <= 1.0


# Portfolio depth check

**Meaningful code lines visible in this notebook:** 425. For a major recruiter-facing application the working target is roughly **1,000 meaningful lines**, with a practical guide of about 600–1,400 depending on the problem. This notebook is below the major-project guide and should grow only through substantive analysis/application depth.

Line count is not a quality metric by itself. Add code only when it improves the real project: data acquisition, validation, cleaning, EDA, visualisation, feature engineering, baselines, model comparison, tuning, leakage control, error analysis, explainability, uncertainty, inference, tests, monitoring, deployment or decision logic.
